# Analisis Trend Klaim Asuransi vs Hari Libur & Weekend

Notebook ini menganalisis hubungan antara data klaim asuransi dengan kalender hari libur nasional dan weekend Indonesia (2024–2025), mencakup:
- **Total Klaim** per kategori hari (hari kerja / weekend / libur nasional)
- **Nominal Klaim** yang disetujui terhadap jenis hari
- **Proses Pencairan** (Processing Days) terhadap jenis hari submission dan pembayaran

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from matplotlib.lines import Line2D
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 10
sns.set_style('whitegrid')
PALETTE = {'Hari Kerja': '#4C72B0', 'Weekend': '#DD8452', 'Libur Nasional': '#55A868'}

## 1. Load & Prepare Data

In [ ]:
# --- Load data klaim ---
df_klaim = pd.read_csv('dataset/Data_Klaim_Enriched.csv', parse_dates=[
    'Tanggal Pembayaran Klaim', 'Tanggal Pasien Masuk RS', 'Tanggal Pasien Keluar RS'
])

# --- Load data kalender libur ---
df_libur = pd.read_csv(
    'dataset/dataset-libur-nasional-dan-weekend.csv',
    header=None,
    names=['date', 'is_holiday', 'holiday_name', 'is_weekend', 'day_name'],
    parse_dates=['date']
)
df_libur = df_libur.dropna(subset=['date'])

# Kategori hari
def kategorikan_hari(row):
    if row['is_holiday'] == 1 and row['is_weekend'] == 0:
        return 'Libur Nasional'
    elif row['is_holiday'] == 1 and row['is_weekend'] == 1:
        return 'Libur Nasional'  # libur yang jatuh di weekend tetap dihitung libur nasional
    elif row['is_weekend'] == 1:
        return 'Weekend'
    else:
        return 'Hari Kerja'

df_libur['kategori_hari'] = df_libur.apply(kategorikan_hari, axis=1)
df_libur['is_non_working'] = ((df_libur['is_holiday'] == 1) | (df_libur['is_weekend'] == 1)).astype(int)

print('Shape klaim:', df_klaim.shape)
print('Shape kalender:', df_libur.shape)
print('\nKategori hari:')
print(df_libur['kategori_hari'].value_counts())

In [ ]:
# --- Merge klaim dengan kalender berdasarkan tanggal MASUK RS ---
df_masuk = df_klaim.merge(
    df_libur[['date', 'is_holiday', 'is_weekend', 'is_non_working', 'kategori_hari', 'holiday_name', 'day_name']],
    left_on='Tanggal Pasien Masuk RS', right_on='date', how='left'
).rename(columns={
    'is_holiday': 'masuk_is_holiday',
    'is_weekend': 'masuk_is_weekend',
    'is_non_working': 'masuk_is_non_working',
    'kategori_hari': 'masuk_kategori',
    'holiday_name': 'masuk_holiday_name',
    'day_name': 'masuk_day_name'
}).drop(columns=['date'])

# --- Merge berdasarkan tanggal PEMBAYARAN ---
df_bayar = df_masuk.merge(
    df_libur[['date', 'is_holiday', 'is_weekend', 'is_non_working', 'kategori_hari', 'holiday_name', 'day_name']],
    left_on='Tanggal Pembayaran Klaim', right_on='date', how='left'
).rename(columns={
    'is_holiday': 'bayar_is_holiday',
    'is_weekend': 'bayar_is_weekend',
    'is_non_working': 'bayar_is_non_working',
    'kategori_hari': 'bayar_kategori',
    'holiday_name': 'bayar_holiday_name',
    'day_name': 'bayar_day_name'
}).drop(columns=['date'])

df = df_bayar.copy()
print('Dataset gabungan:', df.shape)
df[['Tanggal Pasien Masuk RS', 'masuk_kategori', 'Tanggal Pembayaran Klaim', 'bayar_kategori', 'Processing_Days']].head(5)

## 2. Overview: Distribusi Klaim per Kategori Hari

In [ ]:
# Jumlah klaim berdasarkan hari masuk RS
total_by_hari = df.groupby('masuk_kategori').agg(
    total_klaim=('Claim ID', 'count'),
    total_nominal=('Nominal Klaim Yang Disetujui', 'sum'),
    avg_nominal=('Nominal Klaim Yang Disetujui', 'mean'),
    avg_processing=('Processing_Days', 'mean')
).reset_index()

# Hitung jumlah hari per kategori (untuk normalisasi)
hari_count = df_libur['kategori_hari'].value_counts().reset_index()
hari_count.columns = ['masuk_kategori', 'jumlah_hari']
total_by_hari = total_by_hari.merge(hari_count, on='masuk_kategori')
total_by_hari['klaim_per_hari'] = total_by_hari['total_klaim'] / total_by_hari['jumlah_hari']
total_by_hari['nominal_per_hari'] = total_by_hari['total_nominal'] / total_by_hari['jumlah_hari']

print(total_by_hari[['masuk_kategori','total_klaim','jumlah_hari','klaim_per_hari','avg_nominal','avg_processing']].to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
order = ['Hari Kerja', 'Weekend', 'Libur Nasional']
colors = [PALETTE[k] for k in order]

# Panel 1: Total Klaim
ax1 = axes[0]
data1 = total_by_hari.set_index('masuk_kategori').reindex(order)
bars = ax1.bar(order, data1['total_klaim'], color=colors, edgecolor='white', linewidth=0.8)
for bar, v in zip(bars, data1['total_klaim']):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20, f'{v:,.0f}',
             ha='center', va='bottom', fontsize=9, fontweight='bold')
ax1.set_title('Total Klaim\n(berdasarkan Tgl Masuk RS)', fontweight='bold')
ax1.set_ylabel('Jumlah Klaim')
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

# Panel 2: Rata-rata Nominal Klaim
ax2 = axes[1]
bars2 = ax2.bar(order, data1['avg_nominal']/1e6, color=colors, edgecolor='white', linewidth=0.8)
for bar, v in zip(bars2, data1['avg_nominal']):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3, f'Rp {v/1e6:.1f}M',
             ha='center', va='bottom', fontsize=9, fontweight='bold')
ax2.set_title('Rata-rata Nominal Klaim\n(berdasarkan Tgl Masuk RS)', fontweight='bold')
ax2.set_ylabel('Nominal (Juta Rp)')

# Panel 3: Rata-rata Processing Days
ax3 = axes[2]
bars3 = ax3.bar(order, data1['avg_processing'], color=colors, edgecolor='white', linewidth=0.8)
for bar, v in zip(bars3, data1['avg_processing']):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3, f'{v:.1f} hari',
             ha='center', va='bottom', fontsize=9, fontweight='bold')
ax3.set_title('Rata-rata Proses Pencairan\n(Processing Days)', fontweight='bold')
ax3.set_ylabel('Hari')

plt.suptitle('Kinerja Klaim Asuransi berdasarkan Kategori Hari Masuk RS', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 3. Trend Bulanan: Total Klaim & Nominal

In [ ]:
# Aggregate bulanan
df['bulan_masuk'] = df['Tanggal Pasien Masuk RS'].dt.to_period('M')
df['bulan_bayar'] = df['Tanggal Pembayaran Klaim'].dt.to_period('M')

# Klaim masuk per bulan & kategori hari
bulanan = df.groupby(['bulan_masuk', 'masuk_kategori']).agg(
    total_klaim=('Claim ID', 'count'),
    total_nominal=('Nominal Klaim Yang Disetujui', 'sum')
).reset_index()
bulanan['bulan_str'] = bulanan['bulan_masuk'].astype(str)

# Pivot untuk plot
pivot_klaim = bulanan.pivot_table(index='bulan_str', columns='masuk_kategori', values='total_klaim', fill_value=0)
pivot_nominal = bulanan.pivot_table(index='bulan_str', columns='masuk_kategori', values='total_nominal', fill_value=0)

# Hitung jumlah libur nasional per bulan
df_libur['bulan'] = df_libur['date'].dt.to_period('M').astype(str)
libur_per_bulan = df_libur[df_libur['is_holiday'] == 1].groupby('bulan').size().reset_index(name='n_libur')

print('Bulan tersedia:', sorted(pivot_klaim.index.tolist()))

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 10), sharex=True)

x = range(len(pivot_klaim.index))
labels = list(pivot_klaim.index)

# --- Panel atas: Total Klaim ---
ax1 = axes[0]
width = 0.28
cats = [c for c in ['Hari Kerja', 'Weekend', 'Libur Nasional'] if c in pivot_klaim.columns]
offsets = np.linspace(-width, width, len(cats))

for cat, offset in zip(cats, offsets):
    ax1.bar([xi + offset for xi in x], pivot_klaim[cat], width=width,
            label=cat, color=PALETTE[cat], alpha=0.85, edgecolor='white')

# Annotasi libur nasional
ax1_twin = ax1.twinx()
libur_vals = [libur_per_bulan.set_index('bulan')['n_libur'].get(lbl, 0) for lbl in labels]
ax1_twin.plot(x, libur_vals, 'D--', color='crimson', markersize=5, linewidth=1.5, label='Jml Libur Nasional')
ax1_twin.set_ylabel('Jumlah Hari Libur Nasional', color='crimson', fontsize=9)
ax1_twin.tick_params(axis='y', labelcolor='crimson')
ax1_twin.set_ylim(0, max(libur_vals) * 3)

ax1.set_ylabel('Jumlah Klaim')
ax1.set_title('Trend Bulanan: Total Klaim Masuk RS vs Hari Libur', fontweight='bold')
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax1_twin.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=8)

# --- Panel bawah: Total Nominal ---
ax2 = axes[1]
for cat, offset in zip(cats, offsets):
    ax2.bar([xi + offset for xi in x], pivot_nominal[cat]/1e9, width=width,
            label=cat, color=PALETTE[cat], alpha=0.85, edgecolor='white')

ax2_twin = ax2.twinx()
ax2_twin.plot(x, libur_vals, 'D--', color='crimson', markersize=5, linewidth=1.5, label='Jml Libur Nasional')
ax2_twin.set_ylabel('Jumlah Hari Libur Nasional', color='crimson', fontsize=9)
ax2_twin.tick_params(axis='y', labelcolor='crimson')
ax2_twin.set_ylim(0, max(libur_vals) * 3)

ax2.set_ylabel('Total Nominal (Miliar Rp)')
ax2.set_title('Trend Bulanan: Total Nominal Klaim vs Hari Libur', fontweight='bold')
ax2.set_xticks(x)
ax2.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
lines1, labels1 = ax2.get_legend_handles_labels()
lines2, labels2 = ax2_twin.get_legend_handles_labels()
ax2.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=8)

plt.tight_layout()
plt.show()

## 4. Analisis Processing Days (Proses Pencairan)

In [ ]:
df_proc = df[df['Processing_Days'].notna() & (df['Processing_Days'] <= 200)].copy()

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# --- Boxplot: Processing Days per kategori hari MASUK RS ---
ax1 = axes[0]
order = ['Hari Kerja', 'Weekend', 'Libur Nasional']
data_box = [df_proc[df_proc['masuk_kategori'] == k]['Processing_Days'].dropna() for k in order]
bp = ax1.boxplot(data_box, labels=order, patch_artist=True, notch=False,
                 medianprops={'color': 'black', 'linewidth': 2})
for patch, cat in zip(bp['boxes'], order):
    patch.set_facecolor(PALETTE[cat])
    patch.set_alpha(0.75)

# Tambahkan mean sebagai titik
means = [d.mean() for d in data_box]
ax1.scatter(range(1, len(order)+1), means, marker='D', color='white', edgecolor='black', zorder=5, s=40)
for i, (m, cat) in enumerate(zip(means, order)):
    ax1.text(i+1, m+1.5, f'μ={m:.1f}', ha='center', fontsize=8, color='black')

ax1.set_title('Distribusi Processing Days\nberdasarkan Kategori Hari Masuk RS', fontweight='bold')
ax1.set_ylabel('Processing Days')
ax1.set_xlabel('Kategori Hari Masuk RS')

# --- Heatmap: Avg Processing Days per bulan & kategori ---
ax2 = axes[1]
proc_bulanan = df_proc.groupby(['bulan_masuk', 'masuk_kategori'])['Processing_Days'].mean().reset_index()
proc_bulanan['bulan_str'] = proc_bulanan['bulan_masuk'].astype(str)
proc_pivot = proc_bulanan.pivot_table(index='masuk_kategori', columns='bulan_str', values='Processing_Days')
proc_pivot = proc_pivot.reindex(['Hari Kerja', 'Weekend', 'Libur Nasional'])

sns.heatmap(proc_pivot, ax=ax2, cmap='YlOrRd', annot=True, fmt='.0f',
            linewidths=0.5, cbar_kws={'label': 'Avg Processing Days'},
            annot_kws={'size': 7})
ax2.set_title('Heatmap Avg Processing Days\nper Bulan & Kategori Hari', fontweight='bold')
ax2.set_xlabel('Bulan')
ax2.set_ylabel('Kategori Hari')
ax2.set_xticklabels(ax2.get_xticklabels(), rotation=45, ha='right', fontsize=7)

plt.tight_layout()
plt.show()

In [ ]:
# Tabel ringkasan processing days
proc_summary = df_proc.groupby('masuk_kategori')['Processing_Days'].agg(
    count='count', mean='mean', median='median', std='std',
    q25=lambda x: x.quantile(0.25),
    q75=lambda x: x.quantile(0.75)
).round(1).reindex(['Hari Kerja', 'Weekend', 'Libur Nasional'])

print('=== Ringkasan Processing Days berdasarkan Kategori Hari Masuk RS ===')
print(proc_summary.to_string())

## 5. Pengaruh Libur Panjang terhadap Lonjakan Klaim (Window Analysis)

In [ ]:
# Identifikasi periode libur panjang (>= 2 hari libur/non-working berturut-turut)
df_libur_sorted = df_libur.sort_values('date').copy()
df_libur_sorted['group'] = (df_libur_sorted['is_non_working'] != df_libur_sorted['is_non_working'].shift()).cumsum()
long_holiday = df_libur_sorted[df_libur_sorted['is_non_working'] == 1].groupby('group').agg(
    start=('date', 'min'),
    end=('date', 'max'),
    n_days=('date', 'count'),
    names=('holiday_name', lambda x: ', '.join(x.dropna().unique()))
).reset_index(drop=True)
long_holiday = long_holiday[long_holiday['n_days'] >= 2].reset_index(drop=True)

print(f'Ditemukan {len(long_holiday)} periode libur panjang (≥2 hari berturut-turut):')
print(long_holiday[['start','end','n_days','names']].to_string(index=False))

In [ ]:
# Hitung klaim harian
klaim_harian = df.groupby('Tanggal Pasien Masuk RS').agg(
    total_klaim=('Claim ID', 'count'),
    total_nominal=('Nominal Klaim Yang Disetujui', 'sum')
).reset_index().rename(columns={'Tanggal Pasien Masuk RS': 'date'})

# Merge dengan kalender
klaim_harian = klaim_harian.merge(df_libur[['date','kategori_hari','is_non_working']], on='date', how='outer')
klaim_harian = klaim_harian[klaim_harian['date'].notna()].sort_values('date')
klaim_harian['total_klaim'] = klaim_harian['total_klaim'].fillna(0)
klaim_harian['total_nominal'] = klaim_harian['total_nominal'].fillna(0)
klaim_harian['rolling_7d'] = klaim_harian['total_klaim'].rolling(7, min_periods=1).mean()

print('Klaim harian sample:')
klaim_harian.head()

In [ ]:
# Fokus pada data 2024
klaim_2024 = klaim_harian[klaim_harian['date'].dt.year == 2024].copy()
long_2024 = long_holiday[long_holiday['start'].dt.year == 2024]

fig, axes = plt.subplots(2, 1, figsize=(18, 9), sharex=True)

# --- Panel 1: Total klaim harian + rolling avg ---
ax1 = axes[0]
ax1.fill_between(klaim_2024['date'], klaim_2024['total_klaim'], alpha=0.3, color='#4C72B0')
ax1.plot(klaim_2024['date'], klaim_2024['total_klaim'], color='#4C72B0', linewidth=0.6, alpha=0.7)
ax1.plot(klaim_2024['date'], klaim_2024['rolling_7d'], color='#C44E52', linewidth=1.8, label='Rolling 7 hari')

# Shading periode libur panjang
for _, row in long_2024.iterrows():
    ax1.axvspan(row['start'], row['end'], alpha=0.15, color='orange', zorder=0)
    mid = row['start'] + (row['end'] - row['start']) / 2
    ax1.axvline(mid, color='orange', linewidth=0.5, linestyle='--', alpha=0.5)

ax1.set_ylabel('Jumlah Klaim/Hari')
ax1.set_title('Klaim Masuk RS Harian 2024 (area orange = periode libur panjang)', fontweight='bold')
ax1.legend(fontsize=9)

# --- Panel 2: Nominal klaim harian ---
ax2 = axes[1]
ax2.fill_between(klaim_2024['date'], klaim_2024['total_nominal']/1e9, alpha=0.3, color='#55A868')
ax2.plot(klaim_2024['date'], klaim_2024['total_nominal']/1e9, color='#55A868', linewidth=0.7)

nominal_roll = klaim_2024['total_nominal'].rolling(7, min_periods=1).mean()
ax2.plot(klaim_2024['date'], nominal_roll/1e9, color='#C44E52', linewidth=1.8, label='Rolling 7 hari')

for _, row in long_2024.iterrows():
    ax2.axvspan(row['start'], row['end'], alpha=0.15, color='orange', zorder=0)

ax2.set_ylabel('Total Nominal (Miliar Rp)')
ax2.set_title('Nominal Klaim Harian 2024', fontweight='bold')
ax2.set_xlabel('Tanggal')
ax2.legend(fontsize=9)

# Annotasi libur besar
major_holidays = long_2024[long_2024['n_days'] >= 3]
for _, row in major_holidays.iterrows():
    mid = row['start'] + (row['end'] - row['start']) / 2
    short_name = row['names'].split(',')[0][:20]
    ax1.annotate(short_name, xy=(mid, ax1.get_ylim()[1]*0.9),
                 fontsize=7, ha='center', color='darkorange', rotation=30)

fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## 6. Before–During–After Analysis: Efek Libur Panjang

In [ ]:
# Untuk setiap libur panjang (n_days >= 3), hitung rata-rata klaim:
# 7 hari sebelum, saat libur, 7 hari sesudah

results = []
for _, row in long_holiday[long_holiday['n_days'] >= 3].iterrows():
    start, end = row['start'], row['end']
    name = row['names'].split(',')[0][:25]
    n = row['n_days']

    before = klaim_harian[(klaim_harian['date'] >= start - pd.Timedelta(days=7)) &
                          (klaim_harian['date'] < start)]['total_klaim'].mean()
    during = klaim_harian[(klaim_harian['date'] >= start) &
                          (klaim_harian['date'] <= end)]['total_klaim'].mean()
    after = klaim_harian[(klaim_harian['date'] > end) &
                         (klaim_harian['date'] <= end + pd.Timedelta(days=7))]['total_klaim'].mean()

    results.append({'Libur': f"{name}\n({start.strftime('%d %b')})",
                    'n_hari': n,
                    'Sebelum (7hr)': round(before, 1),
                    'Saat Libur': round(during, 1),
                    'Sesudah (7hr)': round(after, 1)})

df_bda = pd.DataFrame(results)
print(df_bda.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

x = np.arange(len(df_bda))
width = 0.28
colors_bda = ['#4C72B0', '#DD8452', '#55A868']

for i, (col, color, lbl) in enumerate(zip(
    ['Sebelum (7hr)', 'Saat Libur', 'Sesudah (7hr)'],
    colors_bda,
    ['7 Hari Sebelum', 'Saat Libur', '7 Hari Sesudah']
)):
    bars = ax.bar(x + (i-1)*width, df_bda[col], width=width,
                  color=color, label=lbl, alpha=0.85, edgecolor='white')

ax.set_xticks(x)
ax.set_xticklabels(df_bda['Libur'], fontsize=8)
ax.set_ylabel('Rata-rata Klaim/Hari')
ax.set_title('Efek Before–During–After Libur Panjang (≥3 hari) terhadap Volume Klaim', fontweight='bold')
ax.legend()

plt.tight_layout()
plt.show()

## 7. Analisis Hari Pembayaran (Proses Pencairan) vs Libur

In [ ]:
# Distribusi Processing Days berdasarkan bulan pembayaran & jumlah libur di bulan tsb
proc_bayar = df_proc.groupby('bulan_bayar').agg(
    avg_proc=('Processing_Days', 'mean'),
    total_klaim=('Claim ID', 'count')
).reset_index()
proc_bayar['bulan_str'] = proc_bayar['bulan_bayar'].astype(str)

proc_bayar = proc_bayar.merge(libur_per_bulan, left_on='bulan_str', right_on='bulan', how='left')
proc_bayar['n_libur'] = proc_bayar['n_libur'].fillna(0)

fig, ax1 = plt.subplots(figsize=(14, 5))
ax2 = ax1.twinx()

bars = ax1.bar(proc_bayar['bulan_str'], proc_bayar['avg_proc'],
               color='#4C72B0', alpha=0.75, label='Avg Processing Days')
ax1.set_ylabel('Rata-rata Processing Days', color='#4C72B0')
ax1.tick_params(axis='y', labelcolor='#4C72B0')

ax2.plot(proc_bayar['bulan_str'], proc_bayar['n_libur'], 'o-',
         color='crimson', linewidth=2, markersize=6, label='Jml Libur Nasional')
ax2.set_ylabel('Jumlah Libur Nasional', color='crimson')
ax2.tick_params(axis='y', labelcolor='crimson')

ax1.set_xlabel('Bulan Pembayaran')
ax1.set_title('Rata-rata Processing Days per Bulan Pembayaran vs Jumlah Libur Nasional', fontweight='bold')
ax1.set_xticklabels(proc_bayar['bulan_str'], rotation=45, ha='right', fontsize=8)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

# Korelasi
corr = proc_bayar['avg_proc'].corr(proc_bayar['n_libur'])
ax1.text(0.98, 0.95, f'Korelasi: {corr:.3f}', transform=ax1.transAxes,
         ha='right', va='top', fontsize=10,
         bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', edgecolor='gray'))

plt.tight_layout()
plt.show()

print(f'\nKorelasi Avg Processing Days vs Jumlah Libur Nasional per bulan: {corr:.4f}')

## 8. Ringkasan & Insight

In [ ]:
# Tabel ringkasan komprehensif
summary = df.groupby('masuk_kategori').agg(
    total_klaim=('Claim ID', 'count'),
    pct_klaim=('Claim ID', lambda x: f"{len(x)/len(df)*100:.1f}%"),
    avg_nominal=('Nominal Klaim Yang Disetujui', 'mean'),
    total_nominal=('Nominal Klaim Yang Disetujui', 'sum'),
    avg_proc=('Processing_Days', 'mean'),
    median_proc=('Processing_Days', 'median')
).reindex(['Hari Kerja', 'Weekend', 'Libur Nasional']).round(1)

summary['avg_nominal_fmt'] = summary['avg_nominal'].apply(lambda x: f'Rp {x/1e6:.1f}M')
summary['total_nominal_fmt'] = summary['total_nominal'].apply(lambda x: f'Rp {x/1e9:.1f}B')

print('=' * 70)
print('RINGKASAN ANALISIS KLAIM vs HARI LIBUR')
print('=' * 70)
print(summary[['total_klaim', 'pct_klaim', 'avg_nominal_fmt',
               'total_nominal_fmt', 'avg_proc', 'median_proc']].to_string())
print('\nKeterangan: avg_proc & median_proc dalam hari')

print('\n--- KEY INSIGHTS ---')
hk = summary.loc['Hari Kerja']
ln = summary.loc['Libur Nasional'] if 'Libur Nasional' in summary.index else None
we = summary.loc['Weekend'] if 'Weekend' in summary.index else None

if ln is not None:
    diff_proc = ln['avg_proc'] - hk['avg_proc']
    diff_nom = (ln['avg_nominal'] - hk['avg_nominal']) / hk['avg_nominal'] * 100
    print(f"1. Rata-rata Processing Days di Libur Nasional lebih {'panjang' if diff_proc > 0 else 'pendek'} "
          f"{abs(diff_proc):.1f} hari dibanding Hari Kerja")
    print(f"2. Rata-rata Nominal Klaim di Libur Nasional {'lebih tinggi' if diff_nom > 0 else 'lebih rendah'} "
          f"{abs(diff_nom):.1f}% dibanding Hari Kerja")
if we is not None:
    diff_we = we['avg_proc'] - hk['avg_proc']
    print(f"3. Rata-rata Processing Days di Weekend lebih {'panjang' if diff_we > 0 else 'pendek'} "
          f"{abs(diff_we):.1f} hari dibanding Hari Kerja")
print(f"4. Korelasi bulanan Avg Processing Days vs Jumlah Libur: {corr:.4f}")